In [2]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langchain-openai faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA

from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader

**Case Study 1: Document Q&A System**

In [4]:
pdf_url = "https://cs229.stanford.edu/main_notes.pdf"

loader = PyPDFLoader(pdf_url)
documents = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

In [6]:
embeddings = HuggingFaceEmbeddings()

vectorstore = FAISS.from_documents(chunks, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
llm = ChatOpenAI(base_url='https://lightning.ai/api/v1/', model="google/gemini-2.5-flash", api_key="2379b91e-9847-4cdc-be45-0eadefd7649a/mohsin416m/fraud-model")

In [8]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

In [9]:
result = qa_chain({"query": "What is supervised learning?"})
print(result["result"])

/tmp/ipykernel_523/1147197734.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": "What is supervised learning?"})


Supervised learning is a type of machine learning where the goal is to learn a function that can predict an output (target variable) based on given input variables. This is achieved by providing a "training set" which consists of pairs of input and their corresponding correct output values.

Formally, given a training set `{(x^(i), y^(i)); i = 1, ..., n}` (where `x^(i)` is the input and `y^(i)` is the known output for the i-th training example), the objective is to learn a function `h : X ↦→ Y` such that `h(x)` is a good predictor for the corresponding value of `y`. The input variables are denoted as `x` (also called input features), and the output or target variable is denoted as `y`.

Examples of supervised learning problems include:

1.  **Predicting House Prices:** Given data like living area (`x`) and corresponding price (`y`) for several houses, the aim is to learn a function to predict the price of other houses based on their living area.
2.  **Email Spam Filtering:** Given a se

**Case Study 2: Code Documentation Assistant**

In [10]:
!wget -O codebase.py https://raw.githubusercontent.com/pallets/flask/main/src/flask/app.py

--2026-03-17 04:56:27--  https://raw.githubusercontent.com/pallets/flask/main/src/flask/app.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 65423 (64K) [text/plain]
Saving to: ‘codebase.py’

codebase.py         100%[===================>]  63.89K  --.-KB/s    in 0.009s  

2026-03-17 04:56:27 (6.78 MB/s) - ‘codebase.py’ saved [65423/65423]



In [11]:
from langchain_text_splitters import Language

In [12]:
loader = TextLoader("codebase.py")
documents = loader.load()

In [13]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=1000,
    chunk_overlap=200
)

chunks = python_splitter.split_documents(documents)

In [14]:
vectorstore2 = FAISS.from_documents(chunks, embeddings)

In [15]:
code_qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore2.as_retriever(),
    chain_type="map_reduce"
)

In [16]:
result = code_qa({"query": "How are routes registered?"})
print(result["result"])

Routes are usually added as URL rules using the `:meth:`@app.route() <route>` decorator. The provided text doesn't offer further details on the registration process beyond this.


**Case Study 3: Research Paper Assistant**

In [17]:
!wget -O research_paper.pdf https://arxiv.org/pdf/2401.04088

--2026-03-17 04:59:26--  https://arxiv.org/pdf/2401.04088
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.131.42, 151.101.3.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2475990 (2.4M) [application/pdf]
Saving to: ‘research_paper.pdf’

research_paper.pdf  100%[===================>]   2.36M  12.1MB/s    in 0.2s    

2026-03-17 04:59:26 (12.1 MB/s) - ‘research_paper.pdf’ saved [2475990/2475990]



In [18]:
loader = PyPDFLoader("research_paper.pdf")
documents = loader.load()

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

In [20]:
for i, chunk in enumerate(chunks):
    chunk.metadata["paper_id"] = "paper_001"
    chunk.metadata["section"] = "introduction"

In [21]:
vectorstore3 = FAISS.from_documents(chunks, embeddings)

In [26]:
retriever3 = vectorstore3.as_retriever(
    search_kwargs={"k": 5, "filter": {"paper_id": "paper_001"}}
)

In [27]:
qa_chain3 = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever3,
    return_source_documents=True
)

In [28]:
result = qa_chain3({"query": "What is Mixtral 8x7B?"})
print(result["result"])

Mixtral 8x7B is a sparse Mixture-of-Experts (MoE) model. Key characteristics include:

*   **Architecture:** It is based on a transformer architecture where the feed-forward blocks are replaced by Mixture-of-Expert layers.
*   **Parameters:**
    *   It uses **13 billion (13B)** active parameters for each token during inference.
    *   It has a total sparse parameter count of **47 billion (47B)**.
*   **Context Length:** It supports a fully dense context length of 32k tokens.
*   **Pretraining:** It is pretrained with multilingual data.
*   **Performance:** Mixtral 8x7B outperforms or matches Llama 2 70B and GPT-3.5 on almost all popular benchmarks, despite using significantly fewer active parameters (5x fewer than Llama 2 70B).
